In [ ]:
from google.colab import drive

# 다시 마운트
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/My\ Drive/"Path to your folder" # Path to your folder 부분 입력

[Errno 2] No such file or directory: 'drive/My Drive/RND/NIA'
/content/drive/My Drive/RND/NIA


In [ ]:
pip install datasets

In [ ]:
disease = "anxiety" # addiction, depression, anxiety 중 택 1

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# GPU 또는 CPU 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from datetime import datetime

# 타임스탬프 출력
print("실행 시각:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# CPU 사양
print("\n1. CPU 사양")
!lscpu

# GPU 사양 (GPU가 연결된 경우에만 출력됩니다)
print("\n2. GPU 사양")
!nvidia-smi

# RAM 사양
print("\n3. RAM 사양")
!free -h

# HDD 용량
print("\n4. HDD 용량")
!df -h

# OS 버전
print("\n5. OS 버전")
!lsb_release -a

# pytorch 버전
print("\n6. 프레임워크 버전")
print(torch.__version__)

실행 시각: 2024-12-10 07:10:08

1. CPU 사양
Architecture:             x86_64
  CPU op-mode(s):         32-bit, 64-bit
  Address sizes:          46 bits physical, 48 bits virtual
  Byte Order:             Little Endian
CPU(s):                   12
  On-line CPU(s) list:    0-11
Vendor ID:                GenuineIntel
  Model name:             Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:           6
    Model:                85
    Thread(s) per core:   2
    Core(s) per socket:   6
    Socket(s):            1
    Stepping:             7
    BogoMIPS:             4400.33
    Flags:                fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 cl
                          flush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc re
                          p_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3
                           fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand
                         

In [ ]:
# CustomBertForSequenceRegression 클래스 정의
class CustomBertForSequenceRegression(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = 1  # 레이블 수 지정
        self.regressor = nn.Linear(config.hidden_size, self.num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]
        pooled_output = sequence_output[:, 0, :]  # 첫 번째 토큰의 출력 사용
        logits = self.regressor(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.MSELoss()
            loss = loss_fct(logits, labels)
        return (loss, logits) if loss is not None else logits

# 데이터셋 토큰화 함수 정의
def tokenize_function(examples):
    return tokenizer(examples['input'], padding='max_length', truncation=True)

# MultiLabelDataCollator 정의
class MultiLabelDataCollator:
    def __call__(self, features):
        batch = {}
        batch['input_ids'] = torch.stack([f['input_ids'] for f in features])
        batch['attention_mask'] = torch.stack([f['attention_mask'] for f in features])
        if 'token_type_ids' in features[0]:
            batch['token_type_ids'] = torch.stack([f['token_type_ids'] for f in features])
        batch['labels'] = torch.stack([torch.tensor(f['label'], dtype=torch.float) for f in features])
        return batch

# CustomTrainer 정의
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if 'num_items_in_batch' in kwargs:
            kwargs.pop('num_items_in_batch')

        # 기존 로직 유지
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs[1] if isinstance(outputs, tuple) else outputs
        loss_fct = nn.MSELoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


# 0과 3사이 가장 가까운 정수로 변환하는 함수 정의
def closest_integer(predictions):
    return min(max(round(predictions), 0), 3)

# 예측 함수
def predict(sentence, model, tokenizer):
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # GPU로 이동
    model.to(device)  # 모델을 GPU로 이동
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs[1] if isinstance(outputs, tuple) else outputs
    prediction = logits.squeeze().tolist()  # 텐서를 리스트로 변환
    prediction = closest_integer(prediction)  # 가장 가까운 정수로 변환
    return prediction


In [ ]:
import os
import json

# 폴더 경로 설정
folder_path = './data/training'  # 폴더 내 training 데이터셋이 있는 경로 설정

# 폴더 내 addiction 관련 JSON 파일 리스트 가져오기
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json') and (disease in f or 'normal' in f)]

# JSON 파일 하나씩 열어서 데이터 가져오기
labels = []
txts = []

for json_file in json_files:
    file_path = os.path.join(folder_path, json_file)

    with open(file_path, 'r', encoding='utf-8') as f:
        js = json.load(f)
        labels.append(js.get(disease, None))
        paragraghs = js.get("paragraph", None)

        sentences = ""
        for token in paragraghs:
            speaker = token.get("paragraph_speaker", "")
            text = token.get("paragraph_text", "")

            token_sentence = f"{speaker}: {text}\n"
            sentences += token_sentence

        txts.append(sentences)

In [ ]:
print(len(labels))
print(len(txts))
assert len(labels) == len(txts)

528
528


In [ ]:
df = pd.DataFrame({
    'filename': json_files,
    'input': txts,
    'label': labels
})
df.to_excel(disease +'_data.xlsx', index=False)

In [ ]:
# 데이터셋 분할
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)  # 0.2 -> None (test 폴더가 별도로 분할되어 있는 경우)

# Hugging Face Dataset 객체로 변환
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# KlueBERT 모델 및 토크나이저 로드
model_name = "klue/bert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = CustomBertForSequenceRegression.from_pretrained(model_name)

# 데이터셋 토큰화
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# 필요하지 않은 컬럼 제거
train_dataset = train_dataset.remove_columns(['input'])
test_dataset = test_dataset.remove_columns(['input'])

# 포맷 설정
train_dataset.set_format('torch')
test_dataset.set_format('torch')

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=100,
    logging_steps=10,
    weight_decay=0.01,
    logging_dir='./logs_anxiety',
)

# Trainer 초기화
data_collator = MultiLabelDataCollator()
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

# 모델 학습
trainer.train()

# 모델 저장
trainer.save_model("./trained_model_kluebert_"+disease)
tokenizer.save_pretrained("./trained_model_kluebert_"+disease)

Some weights of CustomBertForSequenceRegression were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'regressor.bias', 'regressor.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/422 [00:00<?, ? examples/s]

Map:   0%|          | 0/106 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-18-33e39e2bc729>:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch['labels'] = torch.stack([torch.tensor(f['label'], dtype=torch.float) for f in features])
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([16])) that is different to the input size (torch.Size([16, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch,Training Loss,Validation Loss
1,1.272000,1.088250
2,1.087100,1.033618
3,1.148400,0.855612
4,0.983700,0.822033
5,1.208400,0.942137
6,1.152100,0.892304
7,1.105000,1.216328
8,1.331600,0.906047
9,1.180500,0.979015
10,0.958000,0.924194


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([6])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([10])) that is different to the input size (torch.Size([10, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
<ipython-input-18-33e39e2bc729>:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch['labels'] = torch.stack([torch.tensor(f['label'], dtype=torch.float) for

('./30trained_model_kluebert_anxiety/tokenizer_config.json',
 './30trained_model_kluebert_anxiety/special_tokens_map.json',
 './30trained_model_kluebert_anxiety/vocab.txt',
 './30trained_model_kluebert_anxiety/added_tokens.json')

In [ ]:
### Playground

# 평가
trainer.evaluate()

# 예측 예시 (학습된 모델을 사용)
test_sentence = "비도 오고 그래서 네 생각이 났어"  # 테스트할 문장 입력
predicted_numbers = predict(test_sentence, model, tokenizer)
print(f"\nInput: {test_sentence}\nPredicted numbers: {predicted_numbers}")

# 저장된 모델 호출하여 예측 수행
loaded_model = CustomBertForSequenceRegression.from_pretrained("./trained_model_kluebert_"+disease).to(device)
loaded_tokenizer = BertTokenizer.from_pretrained("./trained_model_kluebert_"+disease)

loaded_model.eval()

<ipython-input-18-33e39e2bc729>:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch['labels'] = torch.stack([torch.tensor(f['label'], dtype=torch.float) for f in features])



Input: 비도 오고 그래서 네 생각이 났어. 생각이 나서 그래서 그랬던거지 별 의미 없지. 오늘은 오랜만에 네 생각을 하는 날이야. 일부러 난 너와 내가 담겨 있는 노랠 찾아. 오늘은 슬프거나 우울해도 괜찮은 마음이야
Predicted numbers: 1


CustomBertForSequenceRegression(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=

In [ ]:
import os
import json

# TEST SET 폴더 입장
folder_path = './data/test'   # 폴더 내 training 데이터셋이 있는 경로 설정

# 폴더 내 disease 관련 JSON 파일 리스트 가져오기
test_json_files = [f for f in os.listdir(folder_path) if f.endswith('.json') and (disease in f or 'normal' in f)]

# JSON 파일별 데이터 가져오기
test_labels = []
test_txts = []

for json_file in test_json_files:
    file_path = os.path.join(folder_path, json_file)

    with open(file_path, 'r', encoding='utf-8') as f:
        js = json.load(f)
        test_labels.append(js.get(disease, None))
        paragraghs = js.get("paragraph", None)

        sentences = ""
        for token in paragraghs:
            speaker = token.get("paragraph_speaker", "")
            text = token.get("paragraph_text", "")
            token_sentence = f"{speaker}: {text}\n"
            sentences += token_sentence

        test_txts.append(sentences)

In [ ]:
import pandas as pd

assert len(test_labels) == len(test_txts)

test_df = pd.DataFrame({
    'filename': test_json_files,
    'input': test_txts,
    'original_label': test_labels,
    'original_label_zeroone' : [0 if x == 0 else 1 for x in test_labels]
})

# 예측 결과를 담을 리스트
predicted_labels = []

# 각 문장에 대해 예측 수행
for sentence in test_df['input']:
    predicted_numbers = predict(sentence, loaded_model, loaded_tokenizer)
    predicted_labels.append(predicted_numbers)

# 예측 결과를 새로운 컬럼 'predicted_label'에 추가
test_df['predicted_label'] = predicted_labels
test_df['predicted_label_zeroone'] = [0 if x == 0 else 1 for x in predicted_labels]


68
68


In [ ]:
original_label_zeroone = np.array(test_df['original_label_zeroone'])
predicted_label_zeroone = np.array(test_df['predicted_label_zeroone'])

# 정확도 계산: 예측이 정확한 경우의 수 / 전체 데이터 수
accuracy_zeroone = np.mean(original_label_zeroone == predicted_label_zeroone)

# 정확도 출력
print(disease, "0~1")
print(f"Accuracy: {accuracy_zeroone * 100:.2f}%")

anxiety 0~1
Accuracy: 73.53%
